# Minnesota Food Shelf Exploratory Data Analysis (EDA)

This notebook consolidates data from several sources and presents visualizations that summarize the data.

In [2]:
import pandas as pd
import matplotlib.pyplot as plt

## Merging and Cleaning

This section prepares the data for analysis by combining the population information from `bea_mn_2024.csv` and the food shelf list from `food_shelves_scrapedgeocoded_2026.pkl` into a single `county` table.

### Cleaning the County Population Data

Source: `bea_mn_2024.csv`.

In [ ]:
bea = pd.read_csv("data/bea_mn_2024.csv", header=3)
len(bea)

The BEA table has footnotes that need to be removed.

In [ ]:
# Displaying the footnotes:
with pd.option_context("display.max_colwidth", None):
    display(bea.loc[:, ["GeoFIPS"]].tail(5))

In [ ]:
# Dropping the footnotes:
bea = bea.drop([264, 265, 266, 267, 268])
# Checking the tail:
bea.tail(3)

The BEA table has summary rows for the whole state that need to be removed.

In [ ]:
# Displaying the summary rows:
bea.head(3)

In [ ]:
# Dropping the MN total rows:
bea = bea.drop([0, 1, 2])
# Checking the head:
bea.head(3)

We can pivot the table so that each row is a county.

In [ ]:
# Pivoting the table so that each row is a county:

# Mapping LineCode to column names:
linecode_map = {1.0: "income", 2.0: "population", 3.0: "income_per_capita"}

# Pivoting and renaming columns:
county = (
    bea[bea["LineCode"].isin(linecode_map.keys())]
    .assign(variable=lambda df: df["LineCode"].map(linecode_map))
    .pivot_table(index="GeoName", columns="variable", values="2024", aggfunc="first")
    .reset_index()
    .rename(columns={"GeoName": "county"})
    .rename_axis(None, axis=1)
)[["county", "income", "population", "income_per_capita"]]

county.head(3)

We perform some common sense checks to verify the data are internally consistent.

In [ ]:
# Checking the number of counties correctly equals 87:
print(f"Total number of counties: {len(county)}\n")

# Checking the sums given by the BEA:
# Total MN population according to the BEA: 5793151.0
print(f"Total MN population (in thousands): {county['population'].sum()}")
# Total personal MN income according to the BEA: 437981871.0
print(f"Total MN personal income (thousands of $): {county['income'].sum()}\n")

# Checking the per capita calculation:
aitkin_inc = county.at[0, "income"]
aitkin_pop = county.at[0, "population"]
aitkin_cap = county.at[0, "income_per_capita"]
print(f"Aitkin County income: {aitkin_inc}")
print(f"Aitkin County population: {aitkin_pop}")
print(f"Aitkin County income per capita (from BEA): {aitkin_cap}")
print(
    f"Aitkin County income per capita (calculated): {round((aitkin_inc * 1000.0) / aitkin_pop,0)}"
)

We standardize the county names.

In [ ]:
# Standardizing county names:
county["county"] = county["county"].str.replace(", MN", "")
county[["county"]].sample(5, random_state=42).sort_values("county")

We check if there are non-integer `income` or `population` values, and then cast to int:

In [ ]:
# Checking for non-whole values:
print(
    county.loc[county["income"] % 1 != 0, "income"].count(),
    county.loc[county["population"] % 1 != 0, "population"].count(),
    county.loc[county["income_per_capita"] % 1 != 0, "income_per_capita"].count(),
)

In [ ]:
# Casting to int:
county["income"] = county["income"].astype("int")
county["population"] = county["population"].astype("int")
county["income_per_capita"] = county["income_per_capita"].astype("int")

# Example row:
county.loc[county["county"] == "Ramsey"]

### Cleaning the Food Shelf Data

The data is from `food_shelves_scrapedgeocoded_2026.pkl`. See `food_shelf_list_reconciliation.ipynb` for some preliminary comparisons to other lists.

In [3]:
shelf = pd.read_pickle("data/food_shelves_scrapedgeocoded_2026.pkl")
len(shelf)

549

In [4]:
# Manually adding missing food shelves:

# Adding Comunidades Latinas Unidas En Servicio (CLUES)- Saint Paul
# Adding Second Harvest Northland: Koochiching County
new_rows = pd.DataFrame(
    [
        {
            "name": "Comunidades Latinas Unidas En Servicio (CLUES)- Saint Paul",
            "address": "797 E 7th St, Saint Paul, MN 55106",
            "lat": pd.NA,
            "lng": pd.NA,
            "county": "Ramsey County",
        },
        {
            "name": "Second Harvest Northland: Koochiching County",
            "address": "10370 MN-11, Birchdale, MN, USA",
            "lat": pd.NA,
            "lng": pd.NA,
            "county": "Koochiching County",
        },
    ]
)
shelf = pd.concat([shelf, new_rows], ignore_index=True)
len(shelf)

/var/folders/xn/9fwvzcw14bjfqgqz3dl2x1840000gn/T/ipykernel_85853/1929479413.py:23: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  shelf = pd.concat([shelf, new_rows], ignore_index=True)


551

In [ ]:
# Checking for dulicate food shelf names that require further investigation:
shelf.duplicated(subset="name").sum()

np.int64(1)

In [ ]:
# Investigating the duplicate further:
shelf.loc[shelf.duplicated(subset="name", keep=False)]

,name,address,lat,lng,county
434,Second Harvest Northland: Itasca County,"51272 State Hwy 46, Squaw Lake, MN, USA",47.617820,-94.132358,Itasca
435,Second Harvest Northland: Itasca County,"53714 County Road 146, Inger, MN, USA",47.552979,-93.980671,Itasca


These are two different stops for Second Harvest Heartland's [Mobile Food Pantry Program](https://secondharvestnorthland.org/find-food/mobile-food-pantry-program/).

The Find Help Map considers these to be distinct service locations, and we will too.

In [5]:
# Checking for missing or mangled counties:
shelf.loc[shelf["county"].isna() | (shelf["county"].str.len() < 10), "county"].count()

np.int64(0)

We standardize the county names.

In [6]:
shelf["county"] = shelf["county"].str.replace("County", "")
shelf["county"] = shelf["county"].str.strip()

print(f"Unique county names: {shelf['county'].nunique()}")

Unique county names: 90


Minnesota only has 87 counties - our food shelf list contains food shelves in non-Minnesota counties.

#### Food Shelves outside Minnesota

16 food shelves on our list aren't in Minnesota.

In [32]:
# Display non-Minnesota food shelves:
(
    shelf.loc[~shelf["address"].str.contains("MN")]
    .loc[~shelf["address"].str.contains("Mn")]
    .loc[~shelf["address"].str.contains("Minnesota")]
    .loc[:, ["name", "address", "county"]]
)

,name,address,county
66,Century College Food Pantry,Century College,Washington
101,Deer River Area Food Shelf,Goodall Resource Center,Itasca
106,Dorothy Day Food Pantry West Fargo,"45 21st Ave W, West Fargo, ND 58078, USA",Clay
117,Emergency Food Pantry,"1101 4th Ave N, Fargo, ND 58102, USA",Clay
123,Family Pathways Frederic,"1100 Wisconsin Ave, Frederic, WI 54837, USA",Chisago
129,Family Pathways St. Croix Falls,"2000 U.S. 8, St Croix Falls, WI 54024, USA",Chisago
323,New Life Center,"1902 3rd Ave N, Fargo, ND 58102, USA",Clay
362,Pierce County Food Pantry,"440 N Maple St, Ellsworth, WI 54011, USA",Goodhue
389,Richland Wilkin Food Pantry,"699 8th Ave S, Wahpeton, ND 58075",Wilkin
404,Salvation Army Fargo,"304 Roberts Street, Fargo, ND 58102",Clay


Three of these rows (indexes 66, 101, and 540) are Minnesota food shelves with mangled addresses, but the rest are in Wisconsin and North Dakota. (Minnesota and North Dakota both have a Cass County.)

Our analysis only examines Minnesota counties, but these particular shelves are included on the Food Group's list because they serve Minnesota residents.

Looking at Richland Wilkin Food Pantry (index 389) on the [Find Help Map](https://www.hungersolutions.org/find-help/?fwp_proximity=46.2920705%2C-96.5204806%2C5%2CBreckenridge%252C%2520MN%252056520%252C%2520USA&fwp_categories=food-shelves) is illustrative:

![A screenshot of the Food Group's Find Help Map showing the Richland Wilkin Food Pantry](images/wilkincountyfoodshelf.png)

Wilkin County's closest food shelf is in Richland County, North Dakota.

In [12]:
wilkin_num_shelf = shelf.loc[shelf["county"] == "Wilkin", "name"].count()
print(f"Number of Food Shelves in Wilkin County: {wilkin_num_shelf}")

Number of Food Shelves in Wilkin County: 0


Because the Richland Wilkin Food Pantry is across state lines, it doesn't appear in our query.

The main focus of the analysis is service availability, and this food shelf is available to Wilkin County residents. As such, we want to "count" it in our analysis.

We manually increment the county's food shelf count by altering the shelf's `county`; because we're not using `address` in our analysis, creating this discrepency is acceptable.

The same turns out to be true for 14 of the 16 non-Minnesota food shelves, and so we manually re-assign each one to the appropriate Minnesota county.

In [ ]:
# Re-assigning
# Richland County to Wilkin County:
shelf.loc[shelf["name"] == "Richland Wilkin Food Pantry", "county"] = "Wilkin"

# Cass County (Fargo, ND) to Clay County (Moorhead, MN):
shelf.loc[shelf["name"] == "Dorothy Day Food Pantry West Fargo", "county"] = "Clay"
shelf.loc[shelf["name"] == "Emergency Food Pantry", "county"] = "Clay"
shelf.loc[shelf["name"] == "New Life Center", "county"] = "Clay"
shelf.loc[shelf["name"] == "Salvation Army Fargo", "county"] = "Clay"
shelf.loc[shelf["name"] == "St. Mary’s Food Pantry", "county"] = "Clay"

# Polk County to Chisago County:
shelf.loc[shelf["name"] == "Family Pathways Frederic", "county"] = "Chisago"
shelf.loc[shelf["name"] == "Family Pathways St. Croix Falls", "county"] = "Chisago"

# Pierce County to Goodhue County (Red Wing, MN):
shelf.loc[shelf["name"] == "Pierce County Food Pantry", "county"] = "Goodhue"
shelf.loc[shelf["name"] == "Spring Valley Community Food Pantry", "county"] = "Goodhue"

# Grand Forks County to Polk County:
shelf.loc[shelf["name"] == "Salvation Army Grand Forks Food Shelf", "county"] = "Polk"
shelf.loc[shelf["name"] == "St. Joseph’s Social Care Food Pantry", "county"] = "Polk"

Second Harvest Northland: Douglas County (11523 S Business 53, Solon Springs, WI 54873; Douglas County) is 30+ mi (~40 min) from Duluth, MN. Second Harvest Northland: Iron County (606-607 3rd Avenue North, Hurley, WI 54534; Iron County) is 100+ mi (~2 hr) from Duluth, MN.

It doesn't seem reasonable to consider these available to Minnesota residents, so we drop them from the list.

In [28]:
# Dropping WI Second Harvest Northland locations:
shelf = shelf.drop(
    index=[
        shelf.loc[shelf["name"] == "Second Harvest Northland: Douglas County"].index[0],
        shelf.loc[shelf["name"] == "Second Harvest Northland: Iron County"].index[0],
    ]
)

With all non-Minnesota food shelves re-assigned or deleted, the total number of unique counties should equal 87.

In [34]:
print(f"Unique county names: {shelf['county'].nunique()}")
print(f"Number of food shelves: {len(shelf)}")

Unique county names: 87
Number of food shelves: 549


Finally, we need to compare our county lists to make sure they will merge correctly:

In [ ]:
bea_counties = county["county"].unique()
shelf_counties = shelf["county"].unique()

for c in bea_counties:
    if c not in shelf_counties:
        print(f"{c} is unmatched.")

As a point of interest, this is our last chance with the food shelf-level data, so we'll do a brief exploration of the major players in terms of number of locations served.

In [ ]:
# Fixing an inconsistency in how different Food Shelf In-A-Box locations are named:
shelf["name"] = shelf["name"].str.replace(
    "Foodshelf-In-A-Box", "Food Shelf In-A-Box", regex=False
)

In [ ]:
# Checking the full list of shelves for common organization names, then counting the shelves:
# with pd.option_context("display.max_rows", None):
#    display(shelf.loc[:,'name'])
groups = pd.DataFrame(
    {
        "group": [
            "360 Communities",
            "Bountiful Basket",
            "Campus Cupboard",
            "Catholic Charities",
            "CLUES",
            "Dorothy Day",
            "Family Pathways",
            "Food Shelf In-A-Box",
            "Heaven’s Table Food Shelf",
            "Hope for the Community",
            "MAS MN",
            "Mille Lacs Band of Ojibwe Food Shelf",
            "Neighbors Express",
            "Nutritious U Food Pantry",
            "Outreach Food Shelf Mobile Pantry",
            "Salvation Army",
            "Second Harvest",
            "SEMCAC",
            "The Open Door Mobile Pantry",
            "The Sanneh Foundation",
            "VEAP",
        ]
    }
)

groups["count"] = groups["group"].apply(
    lambda g: shelf["name"].str.contains(g, case=False, regex=False).sum()
)

groups.loc[groups["count"] > 3].sort_values("count", ascending=False).reset_index(
    drop=True
).rename(columns={"group": "Organization", "count": "Shelf Count"})

,Organization,Shelf Count
0,Salvation Army,25
1,Food Shelf In-A-Box,24
2,Second Harvest,17
3,Catholic Charities,10
4,Family Pathways,10
5,Hope for the Community,9
6,The Sanneh Foundation,7
7,Neighbors Express,6
8,360 Communities,5
9,Outreach Food Shelf Mobile Pantry,4


### Merging the Tables

In [ ]:
# copy-pasted, to potentially play with
shelf.loc[shelf["name"].str.contains("Salvation Army")]

In [ ]:
# copy-pasted not fixed
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(shelf.loc[shelf["match"] != "Match", "address"])

In [ ]:
metro_counties = [
    "Anoka",
    "Carver",
    "Dakota",
    "Hennepin",
    "Ramsey",
    "Scott",
    "Washington",
]